# Graph signal and permutation benchmarks

**Role:** Analysis.

Compare saved GIN, ring and ring-size models trained with intact, permuted and constant fate vectors, using their exact recorded validation inputs and transformations. Separate signal panels show validation MSE against depth and organoid size; paired difference plots show how much predictive accuracy changes relative to intact signals on matching organoids, folds and seeds. Detailed numerical results are exported to CSV rather than displayed as long tables. The final inference-only benchmark applies intact-trained GINs across depths to intact and ring-permuted validation neighborhoods, reporting total and regional MSE. It does not retrain models or redraw the training-time permutations.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Choose the input run or cached report and all analysis options here before model loading, inference or plotting. Derived paths and model-dependent defaults are resolved in the following cells; each checkpoint restores its saved data and transformations.

In [ ]:
import numpy as np
import torch

# Input runs, data and destinations
TRAINING_NOTEBOOK = 'graph_controls_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.

# Model selection
MODEL_KEYS = None  # Checkpoints for the retrained-control comparison; None includes all.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.

# Plot settings
SIGNAL_ORDER = ['intact', 'permuted', 'constant']  # Preferred panel order; other signals follow.
SIZE_BINS = 5  # Quantile bins for observed organoid size; repeated edges are dropped.

# Inference-only ring permutation (independent of MODEL_KEYS above)
RUN_RING_PERMUTATION = True  # Evaluate intact-trained GIN checkpoints without retraining.
RING_MODEL_NAME = 'gin'  # Intact full-panel GIN family in the saved run; 'film' also supports this test.
RING_DEPTHS = None  # None: all saved depths of this family; otherwise a list such as [0, 1, 2, 3, 4].
RING_HIDDEN_DIM = None  # Select a width if the saved family contains several widths.
RING_MODEL_SEEDS = None  # None: all saved training seeds; all saved folds are always used.
RING_CENTERS_PER_ORGANOID = None  # None: every validation cell; an integer caps a fixed sample per organoid.
RING_PERMUTATION_REPEATS = 1  # Independent ring shuffles; average squared errors across repeats before aggregation.
RING_PERMUTATION_SEED = 42  # Reproducible centers and shuffles, shared across depths/folds/model seeds.
RING_BATCH_SIZE = 64  # Maximum number of ego graphs built and evaluated together.

# Anatomical region definitions
REGION_SETTINGS = None  # None: inherit training regional settings; a dict overrides graph_regions options.


## Select run and models

Use the model catalog to select complete fold/seed families. For modern training runs, each catalog row also records marker subset and graph-signal treatment.

In [ ]:
import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table, organoid_mse_summary
from src.artifacts.paths import resolve_training_run
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
print(f'Run: {RUN_DIR.name} | {len(run.records)} saved checkpoints')

OUTPUT_DIR = RUN_DIR/'analysis'/'graph_signal_controls'  # Shared destination for depth-wise comparisons.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Evaluate recorded validation data

Compute predictions using the saved transformations, then average squared errors within each organoid. Cells are not treated as independent biological replicates.

In [ ]:
if MODEL_KEYS is None:
    MODEL_KEYS = run.records.key.tolist()  # Checkpoint keys included in the comparison.
frames = []
for key in MODEL_KEYS:
    chosen = run.select(key, device=DEVICE)
    frame = prediction_table(chosen, device=DEVICE)
    record = run.records.set_index('key').loc[key]
    for column in ['name','depth','fold','seed','subset','signal']:
        frame[column] = record[column]
    frame['hidden_dim'] = record.get('hidden_dim', chosen['model'].hidden_dim)
    frame['mask_rate'] = str(record.get('rate', 'none'))
    frame['key'] = key
    frames.append(frame)
cells = pd.concat(frames, ignore_index=True)
keys = ['key','name','depth','fold','seed','subset','signal','hidden_dim','mask_rate','organoid_str','n_cells']
organoids = cells.groupby(keys).squared_error.mean().rename('mse').reset_index()
if 'baseline_squared_error' in cells:
    baseline_scores = cells.groupby(keys).baseline_squared_error.mean().rename('baseline_mse').reset_index()
    organoids = organoids.merge(baseline_scores, on=keys, validate='one_to_one')
    organoids['mse_minus_baseline'] = organoids.mse-organoids.baseline_mse
else:
    print('This historical run does not contain saved baseline predictions.')
summary = organoid_mse_summary(organoids, ['name','subset','signal','hidden_dim','mask_rate','depth'])
summary.to_csv(OUTPUT_DIR/'mse_summary.csv', index=False)
organoids.to_csv(OUTPUT_DIR/'organoid_mse.csv', index=False)
print(f"Compared {organoids.key.nunique()} checkpoints on {organoids.organoid_str.nunique()} validation organoids; tables saved to {OUTPUT_DIR}")


## Compare depth and organoid size

Columns separate input signals; the upper panels show depth and the lower panels show organoid size. Model colors stay consistent across panels. Lines show equal-organoid mean validation MSE; shading is ± one standard error across organoids after averaging repeated seeds/validation appearances within each organoid. Dashed baseline curves use each model's corresponding organoid support and the same SEM calculation (they can overlap when support is identical).

These are descriptive bands conditional on fitted models, not uncertainty from retraining. Fewer than two organoids gives an undefined SEM and no band. Size curves pool the selected depths within each organoid, as in the original comparison; quantile bins are descriptive. Figures and numerical summaries are saved in the analysis output directory.


In [ ]:
# Compute bins from unique organoids, independent of the number of models/seeds.
size_info = organoids[['organoid_str','n_cells']].drop_duplicates('organoid_str').copy()
size_info['size_bin'] = pd.qcut(size_info.n_cells, SIZE_BINS, duplicates='drop')
if size_info.size_bin.isna().all():
    size_info['size_bin'] = 'all'  # A cohort with one unique size still has one plotted point.
organoids = organoids.drop(columns='size_bin', errors='ignore').merge(
    size_info[['organoid_str','size_bin']], on='organoid_str', validate='many_to_one')
available_signals = organoids.signal.drop_duplicates().tolist()
signal_order = [s for s in SIGNAL_ORDER if s in available_signals]
signal_order += [s for s in available_signals if s not in signal_order]
variant_columns = ['name','subset','hidden_dim','mask_rate']
variant_groups = list(organoids.groupby(variant_columns, dropna=False, observed=True))
variant_colors = {label: plt.get_cmap('tab10')(i % 10) for i, (label, _) in enumerate(variant_groups)}
# Include metadata in legends only when it distinguishes selected model variants.
legend_columns = ['name'] + [c for c in variant_columns[1:] if organoids[c].nunique(dropna=False) > 1]
variant_labels = {label: ' / '.join(str(value) if column == 'name' else f'{column}={value}'
                                  for column, value in zip(variant_columns, label) if column in legend_columns)
                  for label, _ in variant_groups}
fig, axes = plt.subplots(2, len(signal_order), figsize=(5 * len(signal_order), 8),
                         squeeze=False, sharex='row', sharey='row')
size_summaries = []
for column, signal in enumerate(signal_order):
    subset = organoids[organoids.signal == signal]
    for label, frame in subset.groupby(variant_columns, dropna=False, observed=True):
        color = variant_colors[label]
        positions = frame[['organoid_str','size_bin','n_cells']].drop_duplicates('organoid_str')
        positions = positions.groupby('size_bin', observed=True).n_cells.median().rename('N')
        for row, grouping in enumerate(['depth', 'size_bin']):
            ax = axes[row, column]
            for value, style, suffix in [('mse','o-',''), ('baseline_mse','--',': baseline')]:
                if value not in frame:
                    continue
                curve = organoid_mse_summary(frame, grouping, value=value)
                if grouping == 'size_bin':
                    curve = curve.join(positions, on='size_bin')
                    x = curve.N
                    size_summaries.append(curve.assign(signal=signal, metric=value, **dict(zip(variant_columns, label))))
                else:
                    x = curve.depth
                ax.plot(x, curve['mean'], style, color=color, label=variant_labels[label] + suffix)
                ax.fill_between(x, curve['mean'] - curve['sem'], curve['mean'] + curve['sem'],
                                color=color, alpha=.12 if suffix else .18)
    axes[0, column].set(title=f'{signal}: depth', xlabel='Depth / ring radius',
                        ylabel='Mean organoid MSE (curvature²)', xticks=sorted(organoids.depth.unique()))
    axes[1, column].set(title=f'{signal}: organoid size', xscale='log', xlabel='Cell count N',
                        ylabel='Mean organoid MSE (curvature²)')
    for ax in axes[:, column]:
        ax.legend(fontsize=8)
fig.suptitle('Validation MSE: mean ± SEM across organoids')
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'mse_by_signal.png', dpi=160, bbox_inches='tight')
pd.concat(size_summaries, ignore_index=True).to_csv(OUTPUT_DIR/'size_mse_summary.csv', index=False)
plt.show()


## Paired effect of signal destruction

Pair intact and altered-input model errors on the same organoid, fold, seed, depth and architecture. Each panel compares one altered signal with intact signals; positive differences mean that the altered signal loses predictive accuracy. Average paired differences over repeated seeds/folds within each organoid before plotting the mean ± SEM across organoids. Only complete pairs enter each contrast, and missing paired support is reported in the panel. Detailed differences and support counts are exported to CSV. This compares separately trained models, not inference-only permutations of one fitted model.


In [ ]:
pair_keys = ['name','subset','depth','hidden_dim','mask_rate','fold','seed','organoid_str']
paired = organoids.pivot(index=pair_keys, columns='signal', values='mse')
control_signals = [signal for signal in signal_order if signal != 'intact']
if 'intact' in paired and control_signals:
    differences = paired[control_signals].subtract(paired.intact, axis=0)
    differences.to_csv(OUTPUT_DIR/'paired_signal_control_differences.csv')
    paired_long = differences.reset_index().melt(id_vars=pair_keys, value_vars=control_signals,
                                               var_name='signal', value_name='mse_difference').dropna(subset=['mse_difference'])
    paired_summary = organoid_mse_summary(paired_long, variant_columns + ['signal','depth'], value='mse_difference')
    paired_summary.to_csv(OUTPUT_DIR/'paired_signal_control_summary.csv', index=False)
    fig, axes = plt.subplots(1, len(control_signals), figsize=(5 * len(control_signals), 4),
                             squeeze=False, sharex=True, sharey=True)
    for ax, signal in zip(axes.ravel(), control_signals):
        frame = paired_summary[paired_summary.signal == signal]
        for label, curve in frame.groupby(variant_columns, dropna=False, observed=True):
            curve = curve.sort_values('depth')
            color = variant_colors[label]
            ax.plot(curve.depth, curve['mean'], 'o-', color=color, label=variant_labels[label])
            ax.fill_between(curve.depth, curve['mean'] - curve['sem'], curve['mean'] + curve['sem'],
                            color=color, alpha=.18)
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set(title=f'{signal} − intact', xlabel='Depth / ring radius',
               ylabel='Paired MSE difference (curvature²)', xticks=sorted(organoids.depth.unique()))
        if frame.empty:
            ax.text(.5, .5, 'No matched intact/control organoids', ha='center', transform=ax.transAxes)
        else:
            ax.legend(fontsize=8)
    fig.suptitle('Change in validation error: paired mean ± SEM across organoids')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'paired_signal_differences.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('Select both intact and control signals for paired contrasts.')


## Inference-only ring-preserving permutation across GIN depths

This reproduces the purpose of Experiment 3 in the original `signal_composition_analysis.ipynb`: apply **the same intact-trained GIN checkpoint** to intact and ring-permuted validation neighborhoods. It is separate from the comparisons above, which evaluate models trained on different signals. Select the family, depths, width, sampling and permutation settings near the top; every saved fold and the selected training seeds are evaluated. The default includes **every validation cell**, replacing the previous five-center sample from one checkpoint.

For each recipient, shuffle whole fate vectors independently within exact-hop rings, keeping the center fate, edges, targets and organoid globals fixed. Build ego graphs through the maximum selected depth, as in the original analysis. Use the same centers and shuffles at all depths; shallower networks simply cannot use the outer rings. Different recipients require separate ego graphs because each defines its own rings. Batches limit memory. Evaluation mode disables dropout and freezes batch normalization. Intact ego predictions are checked against intact full-graph predictions to detect any discrepancy from extracting neighborhoods.

Inverse-transform predictions with each checkpoint's saved target transform and restore its saved baseline offset when residualization was used. Compute physical-curvature squared errors at the recipient cells. Average over permutation repeats per cell, then over cells per organoid/region, then over repeated fitted seeds/folds within each organoid. Plots report mean ± SEM across organoids; these bands condition on the fitted models and sampled permutations.

### Source-based region assignments

Use the training run's regional settings unless explicitly overridden. Derive regions from the original graph crypt distances, projected vertex IDs and segmentation circumference profiles using `graph_regions`; no region labels from another experiment are loaded. With default settings, qualified crypt territories require a supported neck-like profile and sufficient cells: s<0.8 is crypt body, 0.8≤s≤1.2 is neck and s>1.2 is the operational villus proxy. Missing/unqualified annotations remain separate in the exports and still contribute to total MSE; they are not silently labelled villus. Small undetected crypts remain a limitation.


In [ ]:
from src.artifacts.runs import select_depth_records
from src.models.gnn import GINCurvature
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.data.permutation import permute_ego_neighborhood_within_rings
from src.analysis.interventions.perturbation import predict_subgraph_center_distribution
from src.analysis.spatial.regions import graph_regions


def ring_permutation_node_errors(selection, annotations, *, radius, centers_per_organoid=None,
                                 repeats=1, seed=42, batch_size=64, device='cpu'):
    """Notebook-owned paired inference; return center errors and intact extraction checks."""
    network = selection['model']
    if not isinstance(network, GINCurvature) or network.num_layers > radius:
        raise ValueError('Ring permutation requires a GIN/FiLM with depth <= ego radius.')
    if network.target_dim != 1:
        raise ValueError('This benchmark currently expects a single curvature target.')
    for name, value in [('repeats', repeats), ('batch_size', batch_size), ('centers_per_organoid', centers_per_organoid)]:
        if value is None and name == 'centers_per_organoid':
            continue
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
            raise ValueError(f'{name} must be a positive integer (or None for all centers).')
    full = prediction_table(selection, device=device, batch_size=batch_size)
    frames, checks = [], []
    for graph in selection['groups']['val']:
        graph = graph.cpu()
        org = str(graph.organoid_str)
        # Stable across Python sessions, model order, depths, folds and model seeds.
        org_seed = int.from_bytes(hashlib.sha256(org.encode()).digest()[:8], 'little')
        center_rng = np.random.default_rng(np.random.SeedSequence([seed, org_seed, 0]))
        centers = np.arange(len(graph.x))
        if centers_per_organoid is not None and centers_per_organoid < len(centers):
            centers = np.sort(center_rng.choice(centers, centers_per_organoid, replace=False))
        generators = [torch.Generator().manual_seed(int(np.random.SeedSequence(
            [seed, org_seed, repeat + 1]).generate_state(1)[0])) for repeat in range(repeats)]
        truth = full[full.organoid_str == org].set_index('node')
        labels = annotations[org].set_index('node')
        if truth.index.has_duplicates or labels.index.has_duplicates:
            raise ValueError(f'Duplicate node keys for {org}.')
        offsets = np.asarray(selection['baseline_offsets'][org]).reshape(-1)
        max_error = 0.
        for start in range(0, len(centers), batch_size):
            chosen = centers[start:start + batch_size]
            subs = build_ego_subgraphs_for_graph(graph, num_hops=radius, centers=chosen.tolist())
            intact, intact_lv = predict_subgraph_center_distribution(
                subs, network, device=device, batch_size=batch_size, pin_memory=str(device).startswith('cuda'))
            expected = truth.loc[chosen, 'pred_transformed'].to_numpy()
            np.testing.assert_allclose(intact, expected, rtol=2e-4, atol=2e-6,
                err_msg=f'Intact ego/full-graph mismatch for {org}; do not interpret the permutation benchmark.')
            max_error = max(max_error, float(np.max(np.abs(intact - expected))))
            for condition, repeat in [('intact', 0)] + [('ring_permuted', r) for r in range(repeats)]:
                if condition == 'intact':
                    mu, lv = intact, intact_lv
                else:
                    edited = permute_ego_neighborhood_within_rings(
                        subs, radius, inplace=False, generator=generators[repeat])
                    mu, lv = predict_subgraph_center_distribution(
                        edited, network, device=device, batch_size=batch_size, pin_memory=str(device).startswith('cuda'))
                _, physical, _ = selection['transform'].inverse_distribution(
                    None, mu, lv, graphs=subs, center_only=True)
                pred = np.asarray(physical).reshape(-1) + offsets[chosen]
                y = truth.loc[chosen, 'y_true'].to_numpy()
                baseline_error = (truth.loc[chosen, 'baseline_squared_error'].to_numpy()
                                  if 'baseline_squared_error' in truth else np.full(len(chosen), np.nan))
                frames.append(pd.DataFrame(dict(organoid_str=org, node=chosen, n_cells=len(graph.x),
                    region=labels.loc[chosen, 'region'].to_numpy(), condition=condition, repeat=repeat,
                    y_true=y, y_pred=pred, squared_error=(pred-y)**2, baseline_squared_error=baseline_error)))
        checks.append(dict(organoid_str=org, centers=len(centers), max_abs_intact_difference=max_error))
    return pd.concat(frames, ignore_index=True), pd.DataFrame(checks)


### Evaluate the selected depth scan

Load only intact full-panel checkpoints and require complete fold/seed grids at each selected depth. Save region assignments, per-cell errors for each shuffle, per-organoid total/regional MSE and extraction checks. A short checkpoint progress message replaces long score tables. The independent settings file identifies the exact models, source regions and permutation choices.


In [ ]:
if RUN_RING_PERMUTATION:
    if not run.modern:
        raise ValueError('Select a training_results run with saved models, splits and data transformations.')
    candidates = run.records[(run.records.name == RING_MODEL_NAME) &
                             (run.records.signal == 'intact') & (run.records.subset == 'all')]
    depths = sorted(candidates.depth.unique()) if RING_DEPTHS is None else list(RING_DEPTHS)
    if not depths or len(set(depths)) != len(depths):
        raise ValueError('Choose distinct saved depths from an intact GIN family.')
    if any(isinstance(d, bool) or not isinstance(d, (int, np.integer)) or d < 0 for d in depths):
        raise ValueError('Depths must be nonnegative integers.')
    ring_records = pd.concat([select_depth_records(run, int(depth), model_name=RING_MODEL_NAME,
        hidden_dim=RING_HIDDEN_DIM, seeds=RING_MODEL_SEEDS) for depth in sorted(depths)], ignore_index=True)
    if ring_records.hidden_dim.nunique() != 1:
        raise ValueError('Select one hidden width for the depth scan.')
    radius = int(ring_records.depth.max())
    region_settings_path = RUN_DIR/'regional_evaluation'/'settings.json'
    if REGION_SETTINGS is None:
        region_options = json.loads(region_settings_path.read_text()) if region_settings_path.exists() else {}
        region_options.pop('dataset', None)
    else:
        region_options = dict(REGION_SETTINGS)
    region_data_dir = ROOT/'training_data'/run.settings['dataset']
    ring_output = OUTPUT_DIR/'ring_permutation'
    (ring_output/'node_scores').mkdir(parents=True, exist_ok=True)
    (ring_output/'settings.json').write_text(json.dumps(dict(
        model_keys=ring_records.key.tolist(), radius=radius, centers_per_organoid=RING_CENTERS_PER_ORGANOID,
        repeats=RING_PERMUTATION_REPEATS, permutation_seed=RING_PERMUTATION_SEED,
        batch_size=RING_BATCH_SIZE, device=DEVICE, dataset=str(region_data_dir), region_options=region_options), indent=2))
    region_annotations, score_parts, check_parts = {}, [], []
    for number, record in enumerate(ring_records.to_dict('records'), start=1):
        print(f"Ring permutation {number}/{len(ring_records)}: depth={record['depth']}, fold={record['fold']}, seed={record['seed']}", flush=True)
        chosen_model = run.select(record['key'], device=DEVICE)
        for graph in chosen_model['groups']['val']:
            org = str(graph.organoid_str)
            if org not in region_annotations:
                region_annotations[org] = graph_regions(chosen_model['raw_graphs'][org], region_data_dir, **region_options)
        node_errors, checks = ring_permutation_node_errors(chosen_model, region_annotations, radius=radius,
            centers_per_organoid=RING_CENTERS_PER_ORGANOID, repeats=RING_PERMUTATION_REPEATS,
            seed=RING_PERMUTATION_SEED, batch_size=RING_BATCH_SIZE, device=DEVICE)
        node_errors.to_csv(ring_output/'node_scores'/f"{record['key']}.csv.gz", index=False)
        # Average squared errors (not predictions) over permutations before averaging cells.
        node_means = node_errors.groupby(['condition','organoid_str','node','region'], observed=True).agg(
            squared_error=('squared_error','mean'), baseline_squared_error=('baseline_squared_error','mean')).reset_index()
        metrics = dict(n_cells=('node','size'), mse=('squared_error','mean'), baseline_mse=('baseline_squared_error','mean'))
        total = node_means.groupby(['condition','organoid_str'], observed=True).agg(**metrics).reset_index().assign(region='total')
        regions = node_means.groupby(['condition','organoid_str','region'], observed=True).agg(**metrics).reset_index()
        scores = pd.concat([total, regions], ignore_index=True)
        for column in ['key','name','depth','hidden_dim','fold','seed']:
            scores[column] = record[column]
            checks[column] = record[column]
        score_parts.append(scores)
        check_parts.append(checks)
        pd.concat(score_parts, ignore_index=True).to_csv(ring_output/'organoid_mse.csv', index=False)
        pd.concat(check_parts, ignore_index=True).to_csv(ring_output/'intact_extraction_checks.csv', index=False)
        del chosen_model, node_errors
    ring_scores = pd.concat(score_parts, ignore_index=True)
    ring_summary = organoid_mse_summary(ring_scores, ['condition','region','depth'])
    ring_summary.to_csv(ring_output/'mse_summary.csv', index=False)
    assignments = pd.concat(region_annotations.values(), ignore_index=True)
    assignments.to_csv(ring_output/'node_regions.csv.gz', index=False)
    assignments.groupby(['region','annotation_status']).agg(organoids=('organoid_str','nunique'),
        cells=('node','size')).to_csv(ring_output/'region_support.csv')
    print(f'Ring-permutation results saved to {ring_output}')


### Total and regional MSE versus depth

Compare intact inputs (blue) and ring-permuted inputs (orange) for the same intact-trained GINs and recipient cells. The four panels show total, crypt body, neck and villus-proxy MSE. A dashed global baseline is included when saved baseline predictions are available. Depth 0 should be unchanged because the center is preserved; a one-layer GIN should also be invariant to within-ring reordering of its neighbor sum, up to numerical precision. Empty regions are explicitly labelled, not plotted as zero error.


In [ ]:
if RUN_RING_PERMUTATION:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    region_titles = {'total':'Total', 'crypt':'Crypt body', 'neck':'Neck', 'villus':'Villus (distance proxy)'}
    for ax, (region, title) in zip(axes.ravel(), region_titles.items()):
        frame = ring_summary[ring_summary.region == region]
        for condition, color, label in [('intact','tab:blue','Intact input'),
                                         ('ring_permuted','tab:orange','Ring-permuted input')]:
            curve = frame[frame.condition == condition].sort_values('depth')
            if curve.empty:
                continue
            ax.plot(curve.depth, curve['mean'], 'o-', color=color, label=label)
            ax.fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color=color, alpha=.18)
        baseline = ring_scores[(ring_scores.region == region) & (ring_scores.condition == 'intact')]
        if baseline.baseline_mse.notna().any():
            curve = organoid_mse_summary(baseline, 'depth', value='baseline_mse')
            ax.plot(curve.depth, curve['mean'], '--', color='black', label='Global baseline')
            ax.fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color='black', alpha=.12)
        ax.set(title=title, xlabel='GIN depth', ylabel='Mean organoid MSE (curvature²)', xticks=sorted(depths))
        if frame.empty:
            ax.text(.5, .5, 'No qualified validation cells', ha='center', transform=ax.transAxes)
        else:
            ax.legend(fontsize=9)
    fig.suptitle('Intact-trained GIN: intact vs ring-permuted validation inputs\nMean ± SEM across organoids')
    fig.tight_layout()
    fig.savefig(ring_output/'mse_by_depth.png', dpi=160, bbox_inches='tight')
    plt.show()
